# 05. Evaluate and calibrate

A decision model returns probabilities, so accuracy alone is not enough. We also ask whether the probabilities are calibrated: when the model says 0.8, should it be right about 80 percent of the time?

We compute four things: accuracy, Brier score, expected calibration error, and a confidence threshold that escalates uncertain decisions. All code cells are small and end with a validation.


## Step 1. Load the dataset and a client

We use the mock so the notebook is reproducible without a server. Swap in a real client later; the evaluation code does not change.


In [ ]:
from decision_qa import MockClient, read_jsonl
records = read_jsonl('data/pubmedqa_decisions.jsonl')
print(len(records))


In [ ]:
client = MockClient(seed=42, quality=0.75)
print(type(client).__name__)


## Step 2. One result, one probability

`gold_probability` extracts the probability the model assigned to the correct answer. We demonstrate it on a single question before looping.


In [ ]:
from decision_qa.evaluate import gold_probability
record = records[0]
question = record.questions[0]
result = client.decision(record.state, [question])[0]
print(result.probabilities)


In [ ]:
p = gold_probability(result, question)
print(round(p, 3))
assert 0.0 <= p <= 1.0


## Step 3. Collect all predictions

We walk through every record and every question, storing the gold probability and a binary flag that says whether the gold answer was the top answer.


In [ ]:
gold_probs = []
correct = []
for rec in records:
    for q, res in zip(rec.questions, client.decision(rec.state, rec.questions)):
        gp = gold_probability(res, q)
        gold_probs.append(gp)
        correct.append(int(gp >= max(res.probabilities.values()) - 1e-12))


In [ ]:
print(len(gold_probs), len(correct))
assert len(gold_probs) == len(correct)


## Step 4. Accuracy

Accuracy is the share of questions where the gold answer was the top answer. It says nothing about the quality of the probability values.


In [ ]:
acc = sum(correct) / len(correct)
print(round(acc, 3))
assert 0.0 <= acc <= 1.0


## Step 5. Brier score

Brier score is the mean squared error of the gold probabilities. Lower is better. A model that is both accurate and confident scores close to zero.


In [ ]:
from decision_qa import brier
score = brier(gold_probs)
print(round(score, 3))
assert 0.0 <= score <= 1.0


In [ ]:
one = brier([0.8])
print(round(one, 3))
assert abs(one - 0.04) < 1e-9


## Step 6. Reliability and ECE

The reliability table groups predictions by confidence and compares average confidence with observed accuracy. ECE is the weighted gap between the two.


In [ ]:
from decision_qa.evaluate import reliability
rows = reliability(gold_probs, correct)
print(len(rows))


In [ ]:
for row in rows:
    print(row['bin'], row['count'], round(row['confidence'], 2), round(row['accuracy'], 2))


In [ ]:
from decision_qa import ece
error = ece(gold_probs, correct)
print(round(error, 3))
assert 0.0 <= error <= 1.0


## Step 7. A confidence threshold

`fit_threshold` finds the lowest cutoff that reaches a target precision on this data. Decisions below the cutoff should be escalated. In a real project the threshold must be fit on a validation split, never on the test split.


In [ ]:
from decision_qa import fit_threshold
threshold = fit_threshold(gold_probs, correct, target_precision=0.9)
print(round(threshold, 2))


In [ ]:
coverage = sum(p >= threshold for p in gold_probs) / len(gold_probs)
print(round(coverage, 3))
assert 0.0 <= coverage <= 1.0


## Step 8. Option order invariance

Finally we check that a labelled choice keeps the same top answer when options are reordered. This guards against positional bias.


In [ ]:
from decision_qa.evaluate import option_order_invariance
from decision_qa import ChoiceQuestion
q = ChoiceQuestion(label='check', options=['a', 'b', 'c'], gold_index=1)
invariant = option_order_invariance(client, {}, q)
print(invariant)


In [ ]:
assert invariant is True
print('decision is stable under reordering')


## Reading the results critically

High accuracy with poor calibration means the rankings are useful but the probabilities should not be trusted as numbers. Free references worth reading: JevBench under MIT, ProbabilityBench, the paper Evaluating and Benchmarking the System One model, the Early Evidence Audit with its 14-item checklist, and Beyond Calibration. JevAdvBench reminds us that the state is untrusted input, and Jev Does Not Play Dice shows how badly probabilities can drift from reality.
